# Build a consistent AI image pipeline for narrated video series (without an all-in-one subscription)

Many creators producing narrated, faceless video content ("what if" explainers, documentaries,
educational series) reach for all-in-one AI content platforms that bundle script writing, image
generation, and voice cloning behind a monthly subscription. For a single creator producing a
handful of episodes a month, that subscription is often the most expensive part of the pipeline
— while the actual API costs behind it are a few dollars.

This notebook shows a minimal, reproducible pattern to replace that layer:

1. **Claude** turns a narration script into a structured list of per-scene image prompts, with a
   shared style descriptor so every scene stays visually consistent.
2. **Flux (via fal.ai)** renders each scene as a still image, using a deterministic seed strategy
   so re-running a single scene doesn't visually drift from the rest of the episode.
3. A simple **cost comparison** shows the actual per-episode spend versus a typical bundled
   subscription.

This is not specific to any one video style — the pattern generalizes to any narrated,
image-driven content series (explainers, product demos, audiobook-style channels, etc.).

**What you'll need:**
- An `ANTHROPIC_API_KEY` ([console.anthropic.com](https://console.anthropic.com))
- A `FAL_KEY` from [fal.ai](https://fal.ai) (pay-as-you-go, a few dollars of credit is enough to run this notebook)


## Setup

In [1]:
%pip install -q anthropic fal-client requests python-dotenv


/Users/bernhardauer/Documents/claude-cookbooks/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # expects ANTHROPIC_API_KEY and FAL_KEY in a local .env file

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before continuing"
assert os.environ.get("FAL_KEY"), "Set FAL_KEY before continuing"


## Step 1 — Turn a narration script into consistent scene prompts

Instead of manually writing an image prompt per scene, we hand Claude the full narration script
plus a single shared style descriptor, and ask for a structured list of scenes. Using a
[tool call](https://docs.claude.com/en/docs/build-with-claude/tool-use) with a strict JSON schema
keeps the output directly usable by the next step — no prompt parsing required.


In [3]:
import anthropic

client = anthropic.Anthropic()

NARRATION_SCRIPT = """Listen. That sound you're not hearing right now used to be the last thing anyone noticed —
a tap, running whenever you asked for it. Somewhere, right now, someone turns a handle they've
turned ten thousand times before. Nothing comes out.

It never happens with an explosion. It happens with a queue. First the taps in the poorest
districts fall silent. Then hospitals switch to reserves that were never meant to last this long.

Farms are next. No irrigation, no harvest — and a lost harvest this year is a famine next year.
People start looking at the neighbor's well differently.
"""

STYLE_DESCRIPTOR = (
    "cinematic, moody, desaturated cold color grading with a single warm accent light, "
    "high contrast, shallow depth of field, 16:9, photorealistic, dramatic lighting, "
    "no text, no watermark"
)

extract_scenes_tool = {
    "name": "extract_scenes",
    "description": "Break a narration script into an ordered list of visual scenes for image generation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "scenes": {
                "type": "array",
                "minItems": 4,
                "maxItems": 6,
                "items": {
                    "type": "object",
                    "properties": {
                        "id": {"type": "string", "description": "short slug, e.g. '1a_faucet_opener'"},
                        "prompt": {"type": "string", "description": "visual description only, no camera jargon repeated from the style descriptor"},
                    },
                    "required": ["id", "prompt"],
                },
            }
        },
        "required": ["scenes"],
    },
}

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=1500,
    tools=[extract_scenes_tool],
    tool_choice={"type": "tool", "name": "extract_scenes"},
    messages=[{
        "role": "user",
        "content": (
            f"Break this narration into EXACTLY 4 to 6 distinct visual scenes for a "
            f"still-image, Ken-Burns-style video -- no more than 6, even if the script "
            f"has more beats than that; group related sentences into one scene rather "
            f"than one scene per sentence. One image per scene, ordered as in the script.\n\n"
            f"Script:\n{NARRATION_SCRIPT}"
        ),
    }],
)

raw_scenes = next(block.input["scenes"] for block in response.content if block.type == "tool_use")

# Defensive unwrapping: this is normally already the list of {id, prompt}
# objects from the schema. Handle two variant shapes seen in practice: a
# JSON-encoded string, or an extra {"scenes": [...]} wrapper around the list.
if isinstance(raw_scenes, str):
    raw_scenes = json.loads(raw_scenes)
if isinstance(raw_scenes, dict) and "scenes" in raw_scenes:
    raw_scenes = raw_scenes["scenes"]

assert isinstance(raw_scenes, list) and all(isinstance(s, dict) for s in raw_scenes), (
    f"Expected a list of scene objects, got: {raw_scenes!r}"
)

# Safety net: never trust prompt compliance alone for anything that spends money.
# Even with `maxItems` in the schema, cap defensively so a re-run with a longer
# script can't silently trigger dozens of paid image generations.
MAX_SCENES = 6
if len(raw_scenes) > MAX_SCENES:
    print(f"Claude returned {len(raw_scenes)} scenes, capping at {MAX_SCENES} to control cost.")
    raw_scenes = raw_scenes[:MAX_SCENES]

scenes = [{"id": s["id"], "prompt": s["prompt"]} for s in raw_scenes]

for s in scenes:
    print(f"{s['id']:<20} {s['prompt'][:80]}...")


1_silent_tap         Close-up of an old metal kitchen faucet in dim light, a hand turning the worn ha...
2_hand_turning_handle Weathered hand gripping a rusted outdoor tap handle in a dusty courtyard, turnin...
3_poor_district_silence Narrow alley in a poor urban district lined with communal taps, all dry, people ...
4_hospital_reserves  Dim hospital back room with rows of large water reserve tanks, gauges reading lo...
5_dry_farmland       Vast cracked farmland stretching to the horizon, withered crops in dry irrigatio...
6_neighbors_well     Two neighboring houses separated by a low fence, one figure staring intently acr...


## Step 2 — Render each scene with a consistent, reproducible look

Two things keep a nine-scene episode from looking like it was made by nine different people:

- **A shared style suffix** appended to every prompt (defined once above).
- **A deterministic seed per scene**, derived from one base seed. Re-running a single scene later
  (e.g. after a client review) reproduces the same image instead of a random new one.


In [4]:
import subprocess
import sys
from pathlib import Path

BASE_SEED = 421987
MODEL = "fal-ai/flux-pro/v1.1"
OUT_DIR = Path("scene_images")
OUT_DIR.mkdir(exist_ok=True)

# fal_client's synchronous client can conflict with a Jupyter kernel's own
# event loop in ways that vary by platform/version (deadlocks, silent hangs).
# Running each generation in its own plain Python subprocess sidesteps that
# entirely -- it's the same code path as running a standalone script, just
# invoked once per scene.
_worker_script = Path("_fal_worker.py")
_worker_script.write_text('''import sys
import fal_client
import requests

scene_id, prompt, seed, model, style = sys.argv[1:6]
full_prompt = f"{prompt}, {style}"
result = fal_client.subscribe(
    model,
    arguments={
        "prompt": full_prompt,
        "seed": int(seed),
        "image_size": "landscape_16_9",
        "num_images": 1,
        "output_format": "png",
    },
)
image_url = result["images"][0]["url"]
out_path = f"scene_images/{scene_id}.png"
with open(out_path, "wb") as f:
    f.write(requests.get(image_url, timeout=60).content)
print(out_path)
''')

for i, scene in enumerate(scenes):
    seed = BASE_SEED + i
    print(f"generating {scene['id']} (seed={seed})...")
    result = subprocess.run(
        [sys.executable, str(_worker_script), scene["id"], scene["prompt"], str(seed), MODEL, STYLE_DESCRIPTOR],
        capture_output=True,
        text=True,
        timeout=120,
    )
    if result.returncode == 0:
        print(f"  saved {result.stdout.strip()}")
    else:
        print(f"  ERROR on {scene['id']}: {result.stderr[-500:]}")


generating 1_silent_tap (seed=421987)...


  saved scene_images/1_silent_tap.png
generating 2_hand_turning_handle (seed=421988)...


  saved scene_images/2_hand_turning_handle.png
generating 3_poor_district_silence (seed=421989)...


  ERROR on 3_poor_district_silence: ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bernhardauer/Documents/claude-cookbooks/.venv/lib/python3.12/site-packages/fal_client/client.py", line 1021, in _request
    _raise_for_status(response)
  File "/Users/bernhardauer/Documents/claude-cookbooks/.venv/lib/python3.12/site-packages/fal_client/client.py", line 643, in _raise_for_status
    raise FalClientHTTPError(
fal_client.client.FalClientHTTPError: User is locked. Reason: Exhausted balance. Top up your balance at fal.ai/dashboard/billing.

generating 4_hospital_reserves (seed=421990)...


  saved scene_images/4_hospital_reserves.png
generating 5_dry_farmland (seed=421991)...


  saved scene_images/5_dry_farmland.png
generating 6_neighbors_well (seed=421992)...


  saved scene_images/6_neighbors_well.png


## Step 3 — What this actually costs

Bundled AI content platforms typically charge a flat monthly subscription regardless of how many
episodes you actually produce. The pipeline above only costs what you use.


In [5]:
# Flux Pro v1.1 pricing and Claude API pricing change over time —
# check https://fal.ai/pricing and https://claude.com/pricing for current rates.
FLUX_COST_PER_IMAGE = 0.05   # illustrative, verify current rate
IMAGES_PER_EPISODE = len(scenes)
CLAUDE_COST_PER_EPISODE = 0.05  # a handful of API calls for scene extraction, illustrative

episode_cost = IMAGES_PER_EPISODE * FLUX_COST_PER_IMAGE + CLAUDE_COST_PER_EPISODE
print(f"Images per episode:        {IMAGES_PER_EPISODE}")
print(f"Estimated cost per episode: ${episode_cost:.2f}")
print(f"Estimated cost for 4 episodes/month: ${episode_cost * 4:.2f}")
print("Typical bundled subscription tiers for equivalent AI content platforms: $25-100+/month, "
      "regardless of how many episodes you actually ship.")


Images per episode:        6
Estimated cost per episode: $0.35
Estimated cost for 4 episodes/month: $1.40
Typical bundled subscription tiers for equivalent AI content platforms: $25-100+/month, regardless of how many episodes you actually ship.


## Where to go from here

- **Reference-image consistency**: for scenes that must match exactly (e.g. a recurring shot),
  swap the text-to-image endpoint for Flux's image-to-image / Kontext endpoint and pass a prior
  scene as a reference.
- **Script-to-cue mapping**: once you have a recorded voiceover, you can use a forced-alignment
  tool (e.g. `faster-whisper` for word-level timestamps) to generate an edit decision list that
  times each generated image to the exact moment its keyword is spoken — turning this from a
  batch of images into a synced rough cut automatically.
- **Model choice**: this notebook uses Flux for photorealism and a permissive commercial license;
  swap in any other text-to-image API with the same pattern (shared style suffix + deterministic
  seeds) if your visual style calls for it.

This pattern isn't limited to video content — the same "Claude structures the plan, a
specialized API executes each step deterministically" shape applies anywhere a creative pipeline
needs both language understanding and consistent visual output.
